# Notebook 13 — Deployment: FastAPI + Docker

**Settimana:** 13
**Blocco:** 6.1 — Produzione
**Tempo stimato:** 4-5 ore

## Cosa costruirai

- Una API REST completa per DocBot con FastAPI
- Tutti gli endpoint necessari: `/health`, `/ingest`, `/chat`, `/chat/stream`, `/memory`
- Un `Dockerfile` e `.dockerignore` corretti
- Test dell'API con `httpx` (il client HTTP di FastAPI)
- **Sfida finale:** aggiungi l'autenticazione tramite API key a tutti gli endpoint

## Prerequisiti

Settimana 12 completata. DocBot funziona. Docker installato.

```bash
pip install fastapi uvicorn[standard] httpx python-multipart
docker --version   # deve restituire qualcosa
```

---
## Cella 0 — Setup e navigazione al progetto

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Il codice dell'API vive in proyectos/docbot-api/
API_DIR = Path("../proyectos/docbot-api").resolve()
API_DIR.mkdir(exist_ok=True)

print(f"Directory dell'API: {API_DIR}")

# Verifica che Docker sia disponibile
try:
    resultado = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    print(f"Docker: {resultado.stdout.strip()}")
except FileNotFoundError:
    print("Docker non trovato. Installalo da https://docs.docker.com/get-docker/")
    print("Puoi continuare il notebook fino alla cella di Docker.")

---
## Cella 1 — I modelli Pydantic dell'API

Definisci i contratti di request e response prima di scrivere gli endpoint.
Pydantic valida automaticamente che i dati ricevuti abbiano il formato corretto.

In [ ]:
# Scrive i modelli nella directory dell'API
MODELOS_PY = '''
"""models.py — Contratti Pydantic dell'API."""

from pydantic import BaseModel, Field
from typing import Optional


# ── Requests ──────────────────────────────────────────────────────────────────

class ChatRequest(BaseModel):
    pregunta:   str           = Field(..., min_length=1, max_length=2000,
                                       description="Domanda dell'utente")
    user_id:    str           = Field(default="anon", max_length=64,
                                       description="ID dell'utente")
    session_id: Optional[str] = Field(default=None, description="ID di sessione esistente")


class IngestRequest(BaseModel):
    fuentes:   list[dict]    = Field(..., description="Lista di fonti da indicizzare")
    coleccion: str           = Field(default="rag_docs", description="Nome della collezione")

    class Config:
        json_schema_extra = {
            "example": {
                "fuentes": [
                    {"tipo": "wikipedia", "valor": "FastAPI",
                     "titulo": "FastAPI", "categoria": "framework"}
                ]
            }
        }


# ── Responses ─────────────────────────────────────────────────────────────────

class ChatResponse(BaseModel):
    respuesta:    str
    fuentes:      list[str]
    n_chunks:     int
    respondida:   bool
    latencia_ms:  float
    session_id:   str
    faithfulness: Optional[float] = None


class IngestResponse(BaseModel):
    fuentes_ok:      int
    fuentes_error:   int
    chunks_totales:  int
    n_docs_total:    int


class HealthResponse(BaseModel):
    status:    str          # "ok" | "degraded"
    n_docs:    int
    version:   str = "1.0.0"


class MetricsResponse(BaseModel):
    user_id:             str
    n_total:             int
    n_respondidas:       int
    tasa_respuesta:      float
    faithfulness_media:  Optional[float]
    context_rel_media:   Optional[float]


class ErrorResponse(BaseModel):
    error:   str
    detalle: Optional[str] = None
'''

(API_DIR / "models.py").write_text(MODELOS_PY)
print("models.py creato.")

# ── Test dei modelli ───────────────────────────────────────────────────────
sys.path.insert(0, str(API_DIR))
os.chdir(API_DIR)

# Installa FastAPI se non è presente
try:
    from models import ChatRequest, ChatResponse, IngestRequest, HealthResponse
    print("Modelli importati correttamente.")
except ImportError:
    print("Installazione di fastapi...")
    subprocess.run([sys.executable, "-m", "pip", "install", "fastapi", "uvicorn[standard]", "httpx", "-q"])
    from models import ChatRequest, ChatResponse, IngestRequest, HealthResponse

# Prova di validazione
req_valido = ChatRequest(pregunta="What is FastAPI?", user_id="raffaele")
print(f"\nRequest valido: {req_valido.model_dump()}")

try:
    req_invalido = ChatRequest(pregunta="")  # min_length=1 → deve fallire
except Exception as e:
    print(f"Request non valido (corretto): {type(e).__name__}")

---
## Cella 2 — L'applicazione FastAPI completa

Scrivi il file `api.py` con tutti gli endpoint.
Poi avvia il server e prova ogni endpoint manualmente.

In [ ]:
API_PY = '''
"""
api.py — API REST di DocBot.

Avviare con:
    uvicorn api:app --reload --port 8000
"""

import asyncio
import os
import sys
import threading
from contextlib import asynccontextmanager
from pathlib import Path
from typing import AsyncGenerator

from fastapi import FastAPI, HTTPException, Depends, Header
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv

load_dotenv()

# Aggiunge la directory di DocBot al path
sys.path.insert(0, str(Path(__file__).parent.parent / "docbot"))
from docbot import DocBot, RespuestaDocBot

from models import (
    ChatRequest, ChatResponse,
    IngestRequest, IngestResponse,
    HealthResponse, MetricsResponse, ErrorResponse
)


# ── Pool di bot per utente ──────────────────────────────────────────────────

_bots: dict[str, DocBot] = {}
_lock = threading.Lock()

def obtener_bot(user_id: str) -> DocBot:
    """Restituisce il bot dell'utente, creandolo se non esiste."""
    with _lock:
        if user_id not in _bots:
            _bots[user_id] = DocBot(user_id=user_id)
        return _bots[user_id]


# ── Lifespan: azioni all'avvio/arresto ──────────────────────────────────────

@asynccontextmanager
async def lifespan(app: FastAPI):
    # All'avvio: warm-up (istanzia un bot di prova)
    print("DocBot API in avvio...")
    if not os.getenv("OPENAI_API_KEY"):
        print("ATTENZIONE: OPENAI_API_KEY non configurata")
    yield
    # All'arresto: chiude le sessioni aperte
    print("Chiusura delle sessioni...")
    for bot in _bots.values():
        try:
            bot.cerrar_sesion()
        except Exception:
            pass


# ── Applicazione ────────────────────────────────────────────────────────────────

app = FastAPI(
    title="DocBot API",
    description="Chatbot RAG con memoria. Indicizza documenti e risponde a domande.",
    version="1.0.0",
    lifespan=lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # In produzione: lista esplicita di domini
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Endpoint ─────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse, tags=["sistema"])
async def health():
    """Health check. I sistemi di orchestrazione chiamano questo endpoint per
    verificare che il servizio sia vivo prima di inviargli traffico."""
    bot = obtener_bot("_health")
    return HealthResponse(status="ok", n_docs=bot.n_docs)


@app.post("/ingest", response_model=IngestResponse, tags=["documentos"])
async def ingest(req: IngestRequest):
    """Indicizza documenti nel corpus. Supporta Wikipedia, Markdown e testo semplice."""
    bot = obtener_bot("_ingest")   # Bot condiviso per l'ingestione
    try:
        stats = await asyncio.to_thread(bot.ingestar, req.fuentes)
        return IngestResponse(
            fuentes_ok=stats["fuentes_ok"],
            fuentes_error=stats["fuentes_error"],
            chunks_totales=stats["chunks_totales"],
            n_docs_total=bot.n_docs
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat", response_model=ChatResponse, tags=["chat"])
async def chat(req: ChatRequest):
    """Risponde a una domanda usando il corpus indicizzato."""
    bot = obtener_bot(req.user_id)
    try:
        # asyncio.to_thread: esegue la funzione sincrona senza bloccare l'event loop
        resultado: RespuestaDocBot = await asyncio.to_thread(
            bot.responder, req.pregunta
        )
        return ChatResponse(
            respuesta=resultado.respuesta,
            fuentes=resultado.fuentes,
            n_chunks=resultado.n_chunks,
            respondida=resultado.respondida,
            latencia_ms=resultado.latencia_ms,
            session_id=bot.session_id,
            faithfulness=resultado.faithfulness
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat/stream", tags=["chat"])
async def chat_stream(req: ChatRequest):
    """
    Risponde con Server-Sent Events (SSE).
    Il client riceve i token del LLM in tempo reale.

    Formato di ogni evento:
      data: {"token": "..."}\n\n
      data: {"done": true, "fuentes": [...]}\n\n
    """
    import json as _json
    from openai import OpenAI

    cliente_stream = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    async def generar_tokens() -> AsyncGenerator[str, None]:
        try:
            # Recupera il contesto RAG (sincrono, in thread separato)
            bot = obtener_bot(req.user_id)

            # Facciamo prima il retrieval (sincrono)
            from docbot.retrieval import recuperar, reescribir_query, construir_prompt, UMBRAL_RELEVANCIA
            import chromadb
            chroma   = chromadb.PersistentClient(path="./chroma_db")
            coleccion = chroma.get_collection("rag_docs")

            query    = await asyncio.to_thread(reescribir_query, req.pregunta, cliente_stream)
            chunks   = await asyncio.to_thread(recuperar, query, coleccion, cliente_stream, 4)
            relevantes = [c for c in chunks if c.similitud >= UMBRAL_RELEVANCIA]

            if not relevantes:
                yield f"data: {_json.dumps({'token': "I don\'t have enough information to answer this."})}\n\n"
                yield f"data: {_json.dumps({'done': True, 'fuentes': []})}\n\n"
                return

            prompt  = construir_prompt(req.pregunta, relevantes)
            fuentes = list(dict.fromkeys(c.fuente for c in relevantes))

            # Streaming del LLM
            stream = cliente_stream.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Answer using ONLY the provided context. Cite sources."},
                    {"role": "user",   "content": prompt}
                ],
                stream=True, temperature=0.1, max_tokens=600
            )
            for chunk in stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    yield f"data: {_json.dumps({'token': delta})}\n\n"

            yield f"data: {_json.dumps({'done': True, 'fuentes': fuentes})}\n\n"

        except Exception as e:
            yield f"data: {_json.dumps({'error': str(e)})}\n\n"

    return StreamingResponse(generar_tokens(), media_type="text/event-stream")


@app.get("/metrics/{user_id}", response_model=MetricsResponse, tags=["evaluacion"])
async def get_metrics(user_id: str, n_ultimos: int = 10):
    """Restituisce le metriche di qualità della sessione dell'utente."""
    bot = obtener_bot(user_id)
    try:
        metricas = await asyncio.to_thread(bot.evaluar_sesion, n_ultimos)
        return MetricsResponse(
            user_id=user_id,
            n_total=metricas["n_total"],
            n_respondidas=metricas["n_respondidas"],
            tasa_respuesta=metricas["tasa_respuesta"],
            faithfulness_media=metricas.get("faithfulness_media"),
            context_rel_media=metricas.get("context_rel_media")
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.delete("/memory/{user_id}", tags=["memoria"])
async def delete_memory(user_id: str):
    """Cancella tutta la memoria dell'utente (sessioni + memoria semantica)."""
    if user_id in _bots:
        with _lock:
            bot = _bots.pop(user_id, None)
        if bot:
            await asyncio.to_thread(bot.olvidar)
    return {"message": f"Memoria di '{user_id}' cancellata correttamente."}
'''

(API_DIR / "api.py").write_text(API_PY)
print("api.py creato.")
print("\nPer avviarlo manualmente:")
print(f"  cd {API_DIR}")
print("  uvicorn api:app --reload --port 8000")
print("\nDocumentazione automatica su: http://localhost:8000/docs")

---
## Cella 3 — Test dell'API con httpx

FastAPI include `TestClient` (basato su httpx) che permette di testare
gli endpoint senza avviare un server reale. Scrivi test per tutti gli endpoint.

In [ ]:
# IMPORTANTE: questo blocco presuppone che DocBot e le sue dipendenze siano installate
# e che esista una collezione 'rag_docs' in ./chroma_db con almeno qualche chunk.
# Se non hai la collezione, esegui prima il notebook 12 per crearla.

# Crea il file dei test
TESTS_PY = '''
"""
test_api.py — Test di integrazione per l'API di DocBot.

Eseguire con:
    pytest test_api.py -v
"""

import pytest
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)


# ── Test di health ───────────────────────────────────────────────────────────

def test_health_devuelve_ok():
    r = client.get("/health")
    assert r.status_code == 200
    datos = r.json()
    assert datos["status"] == "ok"
    assert "n_docs" in datos
    assert "version" in datos


# ── Test di ingestione ──────────────────────────────────────────────────────────

def test_ingest_wikipedia():
    r = client.post("/ingest", json={
        "fuentes": [
            {"tipo": "wikipedia", "valor": "Python (programming language)",
             "titulo": "Python", "categoria": "linguaggio"}
        ]
    })
    assert r.status_code == 200
    datos = r.json()
    assert datos["fuentes_ok"] == 1
    assert datos["chunks_totales"] > 0


def test_ingest_fuente_invalida():
    """Fonte con tipo sconosciuto: deve essere processata senza errore 500 (fallisce con grazia)."""
    r = client.post("/ingest", json={
        "fuentes": [{"tipo": "pdf_inexistente", "valor": "niente.pdf", "titulo": "test"}]
    })
    assert r.status_code == 200
    datos = r.json()
    assert datos["fuentes_error"] >= 1   # fallita, ma non ha lanciato 500


# ── Test di chat ─────────────────────────────────────────────────────────────

def test_chat_pregunta_dentro_corpus():
    r = client.post("/chat", json={
        "pregunta": "What is FastAPI?",
        "user_id":  "test_user"
    })
    assert r.status_code == 200
    datos = r.json()
    assert "respuesta" in datos
    assert "session_id" in datos
    assert isinstance(datos["respondida"], bool)
    assert datos["latencia_ms"] > 0


def test_chat_pregunta_fuera_corpus():
    r = client.post("/chat", json={
        "pregunta": "What is the current stock price of Apple?",
        "user_id":  "test_user"
    })
    assert r.status_code == 200
    datos = r.json()
    # Può rispondere o meno, ma non deve restituire 500
    assert "respuesta" in datos


def test_chat_pregunta_vacia_falla_validacion():
    r = client.post("/chat", json={"pregunta": "", "user_id": "test_user"})
    assert r.status_code == 422   # Unprocessable Entity — Pydantic rifiuta min_length=1


def test_chat_session_id_devuelto():
    """Due request dello stesso utente devono restituire lo stesso session_id."""
    r1 = client.post("/chat", json={"pregunta": "What is Docker?", "user_id": "user_persistencia"})
    r2 = client.post("/chat", json={"pregunta": "And Kubernetes?",  "user_id": "user_persistencia"})
    assert r1.status_code == 200
    assert r2.status_code == 200
    # Stesso utente → stessa sessione
    assert r1.json()["session_id"] == r2.json()["session_id"]


# ── Test di memoria ──────────────────────────────────────────────────────────

def test_delete_memory():
    # Crea un po' di attività
    client.post("/chat", json={"pregunta": "Hello", "user_id": "borrar_test"})
    # Cancella la memoria
    r = client.delete("/memory/borrar_test")
    assert r.status_code == 200
    assert "cancellata" in r.json()["message"].lower() or "cancellato" in r.json()["message"].lower()


# ── Test di struttura della risposta ─────────────────────────────────────────

def test_chat_response_tiene_todos_los_campos():
    r = client.post("/chat", json={"pregunta": "What is JWT?", "user_id": "campos_test"})
    assert r.status_code == 200
    datos = r.json()
    campos_requeridos = ["respuesta", "fuentes", "n_chunks", "respondida", "latencia_ms", "session_id"]
    for campo in campos_requeridos:
        assert campo in datos, f"Campo mancante: {campo}"
'''

(API_DIR / "test_api.py").write_text(TESTS_PY)
print("test_api.py creato.")

# ── Esegue i test (richiede che l'API e DocBot siano installati) ─────────
print("\nPer eseguire i test:")
print(f"  cd {API_DIR}")
print("  pip install pytest")
print("  pytest test_api.py -v")

# Tentativo di esecuzione diretta (può fallire se mancano dipendenze)
try:
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "test_api.py", "-v", "--tb=short"],
        capture_output=True, text=True, cwd=API_DIR
    )
    print("\n" + resultado.stdout[-3000:] if len(resultado.stdout) > 3000 else "\n" + resultado.stdout)
    if resultado.returncode != 0:
        print("STDERR:", resultado.stderr[-500:])
except Exception as e:
    print(f"Non è stato possibile eseguire pytest automaticamente: {e}")
    print("Esegui i test manualmente dal terminale.")

---
## Cella 4 — Dockerfile e docker-compose

Impacchetta l'API in un container Docker.
Poi costruiscilo e avvialo localmente per verificare che funzioni.

In [ ]:
# ── requirements.txt ──────────────────────────────────────────────────────────
REQUIREMENTS = """fastapi>=0.111.0
uvicorn[standard]>=0.30.0
openai>=1.30.0
anthropic>=0.25.0
chromadb>=0.5.0
langchain-text-splitters>=0.2.0
python-dotenv>=1.0.0
requests>=2.31.0
httpx>=0.27.0
pytest>=8.0.0
"""

# ── Dockerfile ────────────────────────────────────────────────────────────────
DOCKERFILE = """# ── Immagine base: Python 3.11 minimale (~150MB vs ~900MB full)
FROM python:3.11-slim

# Metadati
LABEL maintainer="raffaele@dihdatalife.com"
LABEL version="1.0.0"
LABEL description="DocBot RAG API"

WORKDIR /app

# ── Installa le dipendenze per prime (viene messo in cache se requirements.txt non cambia)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copia il codice del pacchetto docbot
COPY ../docbot ./docbot

# ── Copia l'API
COPY api.py models.py ./

# ── Crea le directory dati che verranno montate come volumi
RUN mkdir -p chroma_db sesiones evaluaciones

# ── Porta dell'applicazione
EXPOSE 8000

# ── Variabile d'ambiente per far sapere a Chroma dove salvare i dati
ENV CHROMA_DB_PATH=/app/chroma_db

# ── Health check: Docker verifica che il container sia vivo ogni 30s
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# ── Avvio: 1 worker (aumenta con --workers N in produzione multi-core)
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
"""

# ── .dockerignore ─────────────────────────────────────────────────────────────
DOCKERIGNORE = """.env
.env.*
chroma_db/
sesiones/
evaluaciones/
__pycache__/
*.pyc
*.pyo
.git/
.gitignore
*.ipynb
notebooks/
README.md
test_*.py
"""

# ── docker-compose.yml (per lo sviluppo locale) ────────────────────────────────
COMPOSE = """version: '3.8'

services:
  docbot-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
    volumes:
      # Dati persistenti FUORI dal container
      - ./chroma_db:/app/chroma_db
      - ./sesiones:/app/sesiones
      - ./evaluaciones:/app/evaluaciones
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
"""

# Scrive tutti i file
(API_DIR / "requirements.txt").write_text(REQUIREMENTS)
(API_DIR / "Dockerfile").write_text(DOCKERFILE)
(API_DIR / ".dockerignore").write_text(DOCKERIGNORE)
(API_DIR / "docker-compose.yml").write_text(COMPOSE)

print("File Docker creati:")
for f in ["requirements.txt", "Dockerfile", ".dockerignore", "docker-compose.yml"]:
    print(f"  {API_DIR / f}")

# ── Mostra il Dockerfile formattato ─────────────────────────────────────────
print("\n" + "=" * 55)
print("DOCKERFILE")
print("=" * 55)
print(DOCKERFILE)

In [ ]:
# ── Istruzioni per costruire e avviare il container ───────────────────────
print("Per costruire e avviare il container:")
print()
print(f"  cd {API_DIR}")
print("  docker build -t docbot-api:1.0 .")
print()
print("  # Opzione A: docker run diretto")
print("  docker run -p 8000:8000 \\")
print("    -e OPENAI_API_KEY=$OPENAI_API_KEY \\")
print("    -v $(pwd)/chroma_db:/app/chroma_db \\")
print("    -v $(pwd)/sesiones:/app/sesiones \\")
print("    docbot-api:1.0")
print()
print("  # Opzione B: docker-compose (più comodo)")
print("  docker-compose up")
print()
print("Verifica che funzioni:")
print("  curl http://localhost:8000/health")
print("  # Deve restituire: {\"status\": \"ok\", ...}")
print()
print("Documentazione interattiva: http://localhost:8000/docs")

# Tentativo di build automatico (può richiedere alcuni minuti la prima volta)
HACER_BUILD = False   # Cambia a True per eseguire il build automaticamente
if HACER_BUILD:
    print("\nCostruzione dell'immagine Docker in corso (può richiedere ~3 minuti)...")
    result = subprocess.run(
        ["docker", "build", "-t", "docbot-api:1.0", "."],
        capture_output=True, text=True, cwd=API_DIR
    )
    print(result.stdout[-2000:])
    if result.returncode == 0:
        print("✓ Immagine costruita correttamente.")
    else:
        print("✗ Errore nel build:")
        print(result.stderr[-1000:])

---
## Sfida finale — Autenticazione tramite API key

Un'API di produzione non può essere aperta a chiunque.
Aggiungi l'autenticazione tramite API key usando una `Dependency` di FastAPI.

Leggi lo scheletro e completa i `# TODO`.

In [ ]:
AUTH_PY = '''
"""
auth.py — Autenticazione tramite API key per l'API di DocBot.

Uso negli endpoint:
    from auth import verificar_api_key

    @app.post("/chat", dependencies=[Depends(verificar_api_key)])
    async def chat(req: ChatRequest): ...
"""

import os
import secrets
from fastapi import HTTPException, Security, status
from fastapi.security import APIKeyHeader

# L'header dove il client invia l'API key
# Esempio: curl -H "X-API-Key: mia-chiave" http://localhost:8000/chat
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


def _cargar_claves_validas() -> set[str]:
    """
    Carica le API key valide dalle variabili d'ambiente.

    TODO: implementa questa funzione.
    Leggi la variabile d'ambiente API_KEYS (separate da virgola) e restituisci un set.
    Se la variabile non esiste, genera una chiave casuale, stampala nel log,
    e restituiscila come unico elemento del set.

    Esempio di variabile d'ambiente:
      API_KEYS=chiave1,chiave2,chiave3
    """
    # TODO
    raise NotImplementedError


# Carica le chiavi all'importazione del modulo (una volta all'avvio)
_CLAVES_VALIDAS = _cargar_claves_validas()


async def verificar_api_key(api_key: str = Security(api_key_header)) -> str:
    """
    Dependency FastAPI. Verifica che il request includa un'API key valida.

    TODO: implementa la verifica.
    - Se api_key è None o vuota: lancia HTTPException 401 (non autenticato)
    - Se api_key non è in _CLAVES_VALIDAS: lancia HTTPException 403 (vietato)
    - Se è valida: restituisce la api_key

    Usa secrets.compare_digest invece di == per confrontare le stringhe
    (evita i timing attack).
    """
    # TODO
    raise NotImplementedError


def generar_nueva_clave() -> str:
    """
    Genera una API key sicura di 32 byte in formato hex.
    Utile per creare chiavi per nuovi client.
    """
    return secrets.token_hex(32)
'''

(API_DIR / "auth.py").write_text(AUTH_PY)
print("auth.py creato (con TODO da implementare).")

# ── Test dell'autenticazione ──────────────────────────────────────────────────
AUTH_TEST = '''
"""
test_auth.py — Test di autenticazione.
Decommenta quando hai implementato auth.py.
"""

import os
import pytest
from fastapi.testclient import TestClient

# Configura una chiave di test prima di importare l'app
os.environ["API_KEYS"] = "test-key-12345"

# from api_con_auth import app   # versione di api.py con auth applicata
# client = TestClient(app)

# def test_sin_api_key_devuelve_401():
#     r = client.post("/chat", json={"pregunta": "test"})
#     assert r.status_code == 401

# def test_clave_invalida_devuelve_403():
#     r = client.post("/chat",
#         json={"pregunta": "test"},
#         headers={"X-API-Key": "chiave-sbagliata"}
#     )
#     assert r.status_code == 403

# def test_clave_valida_devuelve_200():
#     r = client.post("/chat",
#         json={"pregunta": "What is FastAPI?"},
#         headers={"X-API-Key": "test-key-12345"}
#     )
#     assert r.status_code == 200
'''

(API_DIR / "test_auth.py").write_text(AUTH_TEST)
print("test_auth.py creato.")

# ── Suggerimenti ────────────────────────────────────────────────────────────────
print("\nSuggerimenti per implementare auth.py:")
print("""  _cargar_claves_validas():
    raw = os.getenv("API_KEYS", "")
    if raw:
        return {k.strip() for k in raw.split(",") if k.strip()}
    else:
        nueva = secrets.token_hex(32)
        print(f"API_KEYS non configurata. Chiave temporanea: {nueva}")
        return {nueva}

  verificar_api_key():
    if not api_key:
        raise HTTPException(status_code=401, detail="API key richiesta")
    # Confronta con ogni chiave valida usando compare_digest
    for clave_valida in _CLAVES_VALIDAS:
        if secrets.compare_digest(api_key, clave_valida):
            return api_key
    raise HTTPException(status_code=403, detail="API key non valida")
""")

### Estensioni opzionali

1. **Rate limiting:** aggiungi un limite di request per utente al minuto usando `slowapi` (wrapper di Flask-Limiter per FastAPI).

2. **Logging strutturato:** usa `structlog` affinché ogni request generi un log JSON con `user_id`, `latencia_ms`, `n_chunks` e `respondida`. In produzione questi log vanno a Datadog, Grafana Loki o simili.

3. **Deploy su Railway:** [Railway](https://railway.app) distribuisce container Docker automaticamente da GitHub. Puoi avere la tua API pubblica con URL propria in meno di 10 minuti e piano gratuito.

In [ ]:
# Riepilogo di tutti i file creati questa settimana
print("File generati questa settimana:")
print()
for f in sorted(API_DIR.rglob("*")):
    if f.is_file() and not any(p in str(f) for p in ["__pycache__", ".pyc"]):
        print(f"  {f.relative_to(API_DIR.parent)}")

---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
## Prossima settimana

La settimana 14 completa il blocco di produzione:
**deployment nel cloud** (Railway o Render), variabili d'ambiente in produzione,
monitoraggio di base e il ciclo completo da `git push` fino all'URL pubblica.

---
*Notebook 13 — Settimana 13 del Piano di Formazione AI Engineering*